In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import BayesianRidge
from sklearn.utils.class_weight import compute_class_weight
import time
import os
import random
import keras
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
seed = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

set_seed(seed)

In [ ]:
experiments = [
    # plain baseline
    { 'name': '1L-64-ReLU-Adam', 'hidden_layers': 1, 'neurons': [64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # 1 layer
    { 'name': '1L-128-ReLU-Adam', 'hidden_layers': 1, 'neurons': [128], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '1L-256-ReLU-Adam', 'hidden_layers': 1, 'neurons': [256], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # 2 layers
    { 'name': '2L-64-32-ReLU-Adam', 'hidden_layers': 2, 'neurons': [64, 32], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-128-64-ReLU-Adam', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-256-128-ReLU-Adam', 'hidden_layers': 2, 'neurons': [256, 128], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # 3 layers
    { 'name': '3L-128-64-32-ReLU-Adam', 'hidden_layers': 3, 'neurons': [128, 64, 32], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '3L-256-128-64-ReLU-Adam', 'hidden_layers': 3, 'neurons': [256, 128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # activation function
    { 'name': '2L-128-64-Tanh-Adam', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'tanh', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-128-64-ELU-Adam', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'elu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-128-64-Sigmoid-Adam', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'sigmoid', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # dropout rate
    { 'name': '2L-128-64-ReLU-Drop0.1', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.1, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-128-64-ReLU-Drop0.5', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.5, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-3 },

    # optimizer
    { 'name': '2L-128-64-ReLU-SGD', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'sgd', 'lr': 1e-3 },
    { 'name': '2L-128-64-ReLU-RMSprop', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'rmsprop', 'lr': 1e-3 },
    { 'name': '2L-128-64-ReLU-Adamax', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adamax', 'lr': 1e-3 },

    # learning rate
    { 'name': '2L-128-64-ReLU-Adam-LR0.01', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-2 },
    { 'name': '2L-128-64-ReLU-Adam-LR0.0001', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 32, 'optimizer': 'adam', 'lr': 1e-4 },

    # batch size
    { 'name': '2L-128-64-ReLU-Adam-Batch16', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 16, 'optimizer': 'adam', 'lr': 1e-3 },
    { 'name': '2L-128-64-ReLU-Adam-Batch64', 'hidden_layers': 2, 'neurons': [128, 64], 'activation': 'relu', 'dropout_rate': 0.0, 'epochs': 100, 'batch_size': 64, 'optimizer': 'adam', 'lr': 1e-3 },
]

print(f"Total experiments: {len(experiments)}")

## Formatting Functions

In [ ]:
# For readable print
def header_format(title):
  print("\n" + "=" * 55)
  print(f"{title}")
  print("=" * 55)

def print_row(idx, name, train_acc, test_acc, test_auc, test_prec, test_rec, epochs, elapsed, gap, best=False):
    flag = " ★" if best else "  "
    print(f"{idx:<3} {name:<55} "
          f"{train_acc*100:>10.2f}% "
          f"{test_acc*100:>10.2f}% "
          f"{test_auc*100:>10.2f}% "
          f"{test_prec*100:>10.2f}% "
          f"{test_rec*100:>10.2f}% "
          f"{gap*100:>8.2f}% "
          f"{epochs:>7} "
          f"{elapsed:>7.1f}s{flag}")

## Plot Functions

In [ ]:
# Plot functions
def plot_training_history(history, title_suffix=""):
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    fig.suptitle(f'Training History {title_suffix}', fontsize=14, fontweight='bold')

    metrics = [
        ('loss', 'Loss', 'Loss vs Validation Loss'),
        ('accuracy', 'Accuracy', 'Accuracy vs Validation Accuracy'),
        ('auc', 'AUC', 'AUC vs Validation AUC'),
        ('precision', 'Precision', 'Precision vs Validation Precision'),
        ('recall', 'Recall', 'Recall vs Validation Recall'),
    ]

    axes_flat = axes.flatten()

    for idx, (metric, ylabel, title) in enumerate(metrics):
        ax = axes_flat[idx]

        train_key = metric
        val_key   = f'val_{metric}'

        # Check metric exists in history before plotting
        if train_key in history.history:
            ax.plot(history.history[train_key], label='Train')
        if val_key in history.history:
            ax.plot(history.history[val_key],   label='Val')

        ax.set_title(title)
        ax.set_xlabel('Epochs')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Hide the last empty subplot (2x3 grid has 6 slots, we use 5)
    axes_flat[-1].set_visible(False)

    plt.tight_layout()
    plt.show()

def class_accuracy_report(model, class_names, x_test, y_test):
    header_format("Per-Class Accuracy Report")
    preds = np.argmax(model.predict(x_test, verbose=0), axis=1)
    print(f"\n  {'Class':<14} {'Correct':>8} {'Total':>7} {'Accuracy':>10}")
    print(f"  {'─'*44}")
    for c in range(10):
        mask    = y_test == c
        correct = np.sum(preds[mask] == y_test[mask])
        total   = np.sum(mask)
        print(f"  {class_names[c]:<14} {correct:>8} {total:>7} {correct/total:>9.1%}")

In [ ]:
# Loading the dataset
df_original = pd.read_csv('./data/student_data.csv')
print(df_original.head())
print(df_original.shape)

df_original.drop(columns=['School'], inplace=True)

In [ ]:
df_original.replace('?', np.nan, inplace=True)
df_original = df_original.apply(pd.to_numeric, errors='coerce')

df_original.replace('?', np.nan, inplace=True)

In [ ]:
# Target
target_col = 'FirstYearPersistence'

# continuous
continuous_cols = [
    'First Term Gpa',
    'Second Term Gpa',
    'High School Average Mark',
    'Math Score'
]

# categoricals
categorical_cols = [
    'First Language',
    'Funding',
    'school',
    'FastTrack',
    'Coop',
    'Residency',
    'Gender',
    'Previous Education',
    'Age Group',
    'English Grade'
]

cols_to_scale = continuous_cols + categorical_cols 

In [ ]:
# validation of values
numerical_bounds = {
    'First Term Gpa': {'min': 0.0, 'max': 4.5},
    'Second Term Gpa': {'min': 0.0, 'max': 4.5},
    'High School Average Mark': {'min': 0.0, 'max': 100.0},
    'Math Score': {'min': 0.0, 'max': 50.0}
}

for col, bounds in numerical_bounds.items():
    if col in df_original.columns:
        df_original[col] = df_original[col].clip(lower=bounds['min'], upper=bounds['max'])
        print(f"Clipped '{col}' to range {bounds['min']} - {bounds['max']}")

categorical_allowed = {
    'First Language': [1,2,3],
    'Funding': [1,2,3,4,5,6,7,8,9],
    'FastTrack': [1,2],
    'Coop': [1,2],
    'Residency': [1,2],
    'Gender': [1,2,3],
    'Previous Education': [1,2],
    'Age Group': [1,2,3,4,5,6,7,8,9,10],
    'English Grade': [1,2,3,4,5,6,7,8,9,10,11]
}

for col, allowed_values in categorical_allowed.items():
    if col in df_original.columns:
        invalid_mask = ~df_original[col].isin(allowed_values) & df_original[col].notna()
        invalid_count = invalid_mask.sum()
        print(invalid_count)
        df_original.loc[invalid_mask, col] = np.nan
        
        if invalid_count > 0:
            print(f"Wiped {invalid_count} invalid entries in '{col}' to NaN.")

In [ ]:
df_original['hs_avg_missing'] = df_original['High School Average Mark'].isnull().astype(int)
df_original['math_score_missing'] = df_original['Math Score'].isnull().astype(int)
flag_cols = ['hs_avg_missing', 'math_score_missing']

In [ ]:
df = df_original.copy()  # Keep original for the other approaches
missing_counts = df.isnull().sum()
print(missing_counts)
missing_percentages = (missing_counts / len(df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage Missing (%)': missing_percentages.round(2) 
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values(by='Percentage Missing (%)', ascending=False)

print(missing_summary)

## Understanding first what type of imputation are suitable

## IMPUTATION FUNCTIONS

In [ ]:
def simulation_of_missingness(df_temp, column, percentage, is_categorical=False):
    rnd = np.random.RandomState(42)
    df_known = df_temp[df_temp[column].notna()].copy()
    missing_mask = rnd.rand(len(df_known)) < percentage
    true_values  = df_known.loc[missing_mask, column].values

    df_test = df_known.copy()
    df_test.loc[missing_mask, column] = np.nan

    print(f"\nSimulated missing rows: {missing_mask.sum()}")
    if is_categorical:
        print(f"True value distribution: \n{pd.Series(true_values).value_counts()}")
    else:
        print(f"True values for Mean:   {true_values.mean():.4f}")
        print(f"True values for Median: {np.median(true_values):.4f}")

    print('\n')
    
    return df_known, df_test, missing_mask, true_values

# ------------------------------------------------------------------------------------------------------------------------------------------
def mode_imputer(df_temp, column, missing_mask, true_values, is_categorical=False):
    imputer = SimpleImputer(strategy='most_frequent')
    df_mode = df_temp[[column]].copy()
    df_mode_imputed = pd.DataFrame(
        imputer.fit_transform(df_mode),
        columns=df_mode.columns,
        index=df_mode.index
    )

    mode_preds = df_mode_imputed.loc[missing_mask, column].values

    if is_categorical:
        mode_acc = accuracy_score(true_values, mode_preds)
        print(f"Mode Imputation Accuracy: {mode_acc:.4f}")
    else:
        mode_rmse = np.sqrt(mean_squared_error(true_values, mode_preds))
        mode_mae = mean_absolute_error(true_values, mode_preds)
        print(f"Mode Imputation RMSE: {mode_rmse:.4f} with MAE: {mode_mae:.4f}")

# ------------------------------------------------------------------------------------------------------------------------------------------
def median_imputer(df_temp, column, missing_mask, true_values, is_categorical=False):
    imputer = SimpleImputer(strategy='median')
    df_median = df_temp[[column]].copy()
    df_median_imputed = pd.DataFrame(
        imputer.fit_transform(df_median),
        columns=df_median.columns,
        index=df_median.index
    )

    if is_categorical:
        df_median_imputed[column] = df_median_imputed[column].round().astype(int)
        median_preds = df_median_imputed.loc[missing_mask, column].values
        median_acc = accuracy_score(true_values, median_preds)
        print(f"Median Imputation Accuracy: {median_acc:.4f}")
    else:
        median_preds = df_median_imputed.loc[missing_mask, column].values
        median_rmse  = np.sqrt(mean_squared_error(true_values, median_preds))
        median_mae   = mean_absolute_error(true_values, median_preds)
        print(f"Median RMSE: {median_rmse:.4f} with MAE: {median_mae:.4f}")

# ------------------------------------------------------------------------------------------------------------------------------------------
def knn_imputer(df_temp, feature_cols, column, missing_mask, true_values, neighbors=5, weights='uniform', is_categorical=False):
    minmax_scaler = MinMaxScaler()
    df_knn = df_temp[feature_cols].copy()
    df_knn_scaled = pd.DataFrame(
        minmax_scaler.fit_transform(df_knn[feature_cols]),
        columns=feature_cols,
        index=df_knn.index
    )

    knn_imputer = KNNImputer(n_neighbors=neighbors, weights=weights)
    df_knn_imputed = pd.DataFrame(
        knn_imputer.fit_transform(df_knn_scaled),
        columns=feature_cols,
        index=df_knn.index
    )

    df_knn_inversed = pd.DataFrame(
        minmax_scaler.inverse_transform(df_knn_imputed),
        columns=feature_cols,
        index=df_knn.index
    )
    knn_preds = df_knn_inversed.loc[missing_mask, column].round().astype(int).values

    if is_categorical:
        knn_acc = accuracy_score(true_values, knn_preds)
        print(f"KNN Imputation Accuracy ({neighbors} neighbors, {weights} weights): {knn_acc:.4f}")
    else:
        knn_rmse = np.sqrt(mean_squared_error(true_values, knn_preds))
        knn_mae = mean_absolute_error(true_values, knn_preds)
        print(f"KNN Imputation RMSE: {knn_rmse:.4f} with MAE: {knn_mae:.4f}")

# ------------------------------------------------------------------------------------------------------------------------------------------
def rf_imputer(df_temp, feature_cols, column, missing_mask, true_values, is_categorical=False):
    df_rf = df_temp.copy()
    feature_cols_rf = [col for col in feature_cols if col != column]
    observed = df_rf[column].notna()

    X_train_rf = df_rf.loc[observed, feature_cols_rf]
    y_train_rf = df_rf.loc[observed, column]
    X_pred_rf  = df_rf.loc[missing_mask, feature_cols_rf]

    if is_categorical:
        rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_cls.fit(X_train_rf, y_train_rf)
        rf_preds = rf_cls.predict(X_pred_rf)
        rf_acc   = accuracy_score(true_values, rf_preds)
        print(f"RF Classifier Accuracy: {rf_acc:.4f}")
    else:
        rf_cls = RandomForestRegressor(n_estimators=100, random_state=42)
        rf_cls.fit(X_train_rf, y_train_rf)
        rf_preds = rf_cls.predict(X_pred_rf)
        rf_rmse  = np.sqrt(mean_squared_error(true_values, rf_preds))
        rf_mae   = mean_absolute_error(true_values, rf_preds)
        print(f"RF Regressor RMSE: {rf_rmse:.4f} with MAE: {rf_mae:.4f}")

# ------------------------------------------------------------------------------------------------------------------------------------------
def iterative_imputer(df_temp, feature_cols_iter, column, missing_mask, true_values, estimator, estimater_name):
    iter_bay = IterativeImputer(
        estimator=estimator,
        max_iter=20, # 10 is too small to converge
        random_state=42
    )
    df_iter_bay = df_temp[feature_cols_iter].copy()
    df_iter_bay_imputed = pd.DataFrame(
        iter_bay.fit_transform(df_iter_bay),
        columns=feature_cols_iter,
        index=df_iter_bay.index
    )
    iter_bay_preds = df_iter_bay_imputed.loc[missing_mask, column].values
    iter_bay_rmse  = np.sqrt(mean_squared_error(true_values, iter_bay_preds))
    iter_bay_mae   = mean_absolute_error(true_values, iter_bay_preds)
    print(f"IterativeImputer ({estimater_name}) RMSE: {iter_bay_rmse:.4f} with MAE: {iter_bay_mae:.4f}")


## PREDICTION FUNCTIONS

In [ ]:
def apply_rf_imputation(df, feature_cols, column_to_be_imputed, is_categorical=False):
    df_temp = df.copy()
    existing_data = df_temp[column_to_be_imputed].notna()
    missing_data = df_temp[column_to_be_imputed].isna()
    feature_cols_rf = [col for col in feature_cols if col != column_to_be_imputed]
    X_train_rf = df_temp.loc[existing_data, feature_cols_rf]
    y_train_rf = df_temp.loc[existing_data, column_to_be_imputed]

    X_pred_rf = df_temp.loc[missing_data, feature_cols_rf]
    if is_categorical:
        rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_cls.fit(X_train_rf, y_train_rf)
        predicted = rf_cls.predict(X_pred_rf)
        df_temp.loc[missing_data, column_to_be_imputed] = predicted
    else:
        rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
        rf_reg.fit(X_train_rf, y_train_rf)
        predicted = rf_reg.predict(X_pred_rf)
        df_temp.loc[missing_data, column_to_be_imputed] = predicted
    return df_temp

def apply_iterative_imputation(df, feature_cols_iter, column_to_be_imputed):
    df_temp = df.copy()
    iter_bay = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=20,
        tol=1e-2,
        random_state=42
    )

    # Fit and transform only the relevant columns
    df_iter_cols = df_temp[feature_cols_iter].copy()
    df_iter_imputed = pd.DataFrame(
        iter_bay.fit_transform(df_iter_cols),
        columns=feature_cols_iter,
        index=df_iter_cols.index
    )

    # Fill only the specified column back into original df
    df_temp[column_to_be_imputed] = df_iter_imputed[column_to_be_imputed]
    return df_temp

## Approach #1 : Attempt to handle missing data by simulation

In [ ]:
# As the result of above code snippet, 'Age Group' have 0.28 percent of missing data or 4 entries in the dataset.
# Since it is quite small, I'll apply siimple mode imputation to these two columns.
age_group_mode = df['Age Group'].mode()[0]
df['Age Group'] = df['Age Group'].fillna(age_group_mode)

print('\nValidation after imputation:')
print(df['Age Group'].isnull().sum())

In [ ]:
# Same with 'Age Group', the percentage for 'First Term Gpa' is also quite small.
column_to_be_imputed = 'First Term Gpa'
print(df[column_to_be_imputed].describe())
print(f"Mean: {df[column_to_be_imputed].mean():.4f}")
print(f"Median: {df[column_to_be_imputed].median():.4f}")
print(f"Skew: {df[column_to_be_imputed].skew():.4f}")
print(f"Missing count: {df[column_to_be_imputed].isnull().sum()}")

# 'First Term Gpa' is a little skewed, applying median mputation would be more appropriate.
first_term_median = df[column_to_be_imputed].median()
df[column_to_be_imputed] = df[column_to_be_imputed].fillna(first_term_median)
print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())

In [ ]:
# 'English Grade' has around 3.13% of missing data.
# One thing I noted is that this is an ordinal column

# I wanted to compare different imputation methods for 'English Grade'
# This is done so by using the rows where there is actual data and removing the English Grade value and is saved as a true value for comparison...
column_to_be_imputed = 'English Grade'
df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.0313, is_categorical=True) # 3.13% missing data

# mode imputation
mode_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# median imputation
median_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# knn imputation
feature_cols = ['First Term Gpa', 'Age Group', 'English Grade', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender']
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=11, weights='distance', is_categorical=True)

english_grade_mode = df[column_to_be_imputed].mode()[0]
df[column_to_be_imputed] = df[column_to_be_imputed].fillna(english_grade_mode)
print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())

# The results are not that high, with mode having the highest accuracy.
# Still thinking if I should drop it or apply the mode imputation

In [ ]:
# 'Previous Education' has 6.40% missing rows
print(df['Previous Education'].value_counts().sort_index())
print(df['Previous Education'].value_counts(normalize=True))
print(df.groupby('Previous Education')['FirstYearPersistence'].mean())
column_to_be_imputed = 'Previous Education'

feature_cols = ['First Term Gpa', 'Previous Education', 'Age Group', 'English Grade', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender']
df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.0640, is_categorical=True)

# mode imputation
mode_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# knn imputation
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=11, weights='distance', is_categorical=True)

# rf imputation
rf_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# apply rf imputation
df = apply_rf_imputation(df, feature_cols, column_to_be_imputed, is_categorical=True)

In [ ]:
# 'First Language' has around 7.72% of missing data.
column_to_be_imputed = 'First Language'
print(df[column_to_be_imputed].value_counts())
print(df[column_to_be_imputed].value_counts(normalize=True))
print(df.groupby(column_to_be_imputed)[ 'FirstYearPersistence'].mean())

df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.0772, is_categorical=True) # 7.72% missing data

# mode imputation
print('\n')
mode_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# knn imputation
feature_cols = ['First Term Gpa', 'English Grade', 'Previous Education', 'Age Group', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender', column_to_be_imputed]
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=5, weights='distance', is_categorical=True)

# rf classifier imputation
rf_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, is_categorical=True)

# APplying the RF Classifier imputation since it has the highest accuracy
df = apply_rf_imputation(df, feature_cols, column_to_be_imputed, is_categorical=True)

print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())

In [ ]:
# 'Second Term Gpa' has around 11.13% of missing data, which is quite high.
column_to_be_imputed = 'Second Term Gpa'
print(df[column_to_be_imputed].describe())
print(f"\nMean: {df[column_to_be_imputed].mean():.4f}")
print(f"Median: {df[column_to_be_imputed].median():.4f}")
print(f"Skew: {df[column_to_be_imputed].skew():.4f}")
print(f"\nCorrelation with First Term GPA: {df[column_to_be_imputed].corr(df['First Term Gpa']):.4f}")
print(f"Correlation with Persistence: {df[column_to_be_imputed].corr(df['FirstYearPersistence']):.4f}")

feature_cols = ['First Term Gpa', 'English Grade', 'Previous Education', 'Age Group', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender', 'First Language', column_to_be_imputed]

df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.1113, is_categorical=False) # 11.13% missing data

# median imputation
median_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# knn imputation
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=5, weights='distance', is_categorical=False)

# rf regressor imputation
rf_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# Applying RF Regressor imputation since it has the lowest RMSE and MAE
df = apply_rf_imputation(df, feature_cols, column_to_be_imputed, is_categorical=False)

print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())


In [ ]:
# 'Math Score' has around 32.15% of missing data 
column_to_be_imputed = 'Math Score'
feature_cols = ['First Term Gpa', 'English Grade', 'Previous Education', 'Age Group', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender', 'First Language', 'Second Term Gpa', column_to_be_imputed]
feature_cols_iterative = feature_cols + ['High School Average Mark']

print(df[column_to_be_imputed].describe())
print(f"\nMean: {df[column_to_be_imputed].mean():.4f}")
print(f"Median: {df[column_to_be_imputed].median():.4f}")
print(f"Skew: {df[column_to_be_imputed].skew():.4f}")

print(f"\nCorrelation with First Term GPA: {df[column_to_be_imputed].corr(df['First Term Gpa']):.4f}")
print(f"Correlation with Second Term GPA: {df[column_to_be_imputed].corr(df['Second Term Gpa']):.4f}")
print(f"Correlation with Persistence: {df[column_to_be_imputed].corr(df['FirstYearPersistence']):.4f}")

print(f"\nMissing count: {df[column_to_be_imputed].isnull().sum()}")
print(f"Missing %: {df[column_to_be_imputed].isnull().mean()*100:.2f}%")

df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.3215, is_categorical=False) # 32.15% missing data

# median imputation
median_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# knn imputation
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=5, weights='distance', is_categorical=False)

# rf regressor imputation
rf_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# iterative imputer Bayesian Ridge
iterative_imputer(df_test, feature_cols_iterative, column_to_be_imputed, missing_mask, true_values, estimator=BayesianRidge(), estimater_name='BayesianRidge')

# iterative imputer Random Forest
# I tried but it wont converge at all. Skipping this estimator.
#iterative_imputer(df_test, feature_cols_iterative, column_to_be_imputed, missing_mask, true_values, estimator=RandomForestRegressor(n_estimators=100, random_state=42), estimater_name='RandomForestRegressor')

# apply iterative imputer
df = apply_iterative_imputation(df, feature_cols_iterative, column_to_be_imputed)
print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())

In [ ]:
#'High School Average Mark' has around 51.70% of missing data
print(df['High School Average Mark'].describe())
print(f"\nMean: {df['High School Average Mark'].mean():.4f}")
print(f"Median: {df['High School Average Mark'].median():.4f}")
print(f"Skew: {df['High School Average Mark'].skew():.4f}")
print(f"\nCorrelation with First Term GPA: {df['High School Average Mark'].corr(df['First Term Gpa']):.4f}")
print(f"Correlation with Second Term GPA: {df['High School Average Mark'].corr(df['Second Term Gpa']):.4f}")
print(f"Correlation with Math Score: {df['High School Average Mark'].corr(df['Math Score']):.4f}")
print(f"Correlation with Persistence: {df['High School Average Mark'].corr(df['FirstYearPersistence']):.4f}")

column_to_be_imputed = 'High School Average Mark'
feature_cols = ['First Term Gpa', 'English Grade', 'Previous Education', 'Age Group', 'Funding', 'FastTrack', 'Coop', 'Residency', 'Gender', 'First Language', 'Second Term Gpa', 'Math Score', column_to_be_imputed]
df_known, df_test, missing_mask, true_values = simulation_of_missingness(df, column_to_be_imputed, percentage=0.5170, is_categorical=False) # 51.70% missing data

# median imputation
median_imputer(df_test, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# knn imputation
knn_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, neighbors=5, weights='distance', is_categorical=False)

# rf regressor
rf_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, is_categorical=False)

# iterative imputer Bayesian Ridge
iterative_imputer(df_test, feature_cols, column_to_be_imputed, missing_mask, true_values, estimator=BayesianRidge(), estimater_name='BayesianRidge')

# iterative imputer Random Forest
iterative_imputer(df_test, feature_cols_iterative, column_to_be_imputed, missing_mask, true_values, estimator=RandomForestRegressor(n_estimators=100, random_state=42), estimater_name='RandomForestRegressor')

# apply iterative imputer with bayesian ridge
df = apply_iterative_imputation(df, feature_cols_iterative, column_to_be_imputed)
df[column_to_be_imputed] = df[column_to_be_imputed].clip(lower=0.0, upper=100.0)
print('\nValidation after imputation:')
print(df[column_to_be_imputed].isnull().sum())


In [ ]:
# final check
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Model training

## Model Functions

In [ ]:
def get_best_result(results):
    valid_results = [r for r in results if r['test_rec'] < 1.0]
    
    if not valid_results:
        print("WARNING  experiments collapsed to 100% recall!")
        print("Falling back to highest AUC regardless of recall...")
        return max(results, key=lambda x: x['test_auc'])
    
    # Among valid results pick best AUC
    best = max(valid_results, key=lambda x: x['test_auc'])
    collapsed = len(results) - len(valid_results)
    if collapsed > 0:
        print(f"{collapsed} experiment(s) excluded due to 100% recall")
    
    return best

def build_model(input_dim, hidden_layers, neurons, activation, dropout_rate=0.0):
    model = keras.Sequential(name="FirstYearPersistence")
    model.add(layers.Input(shape=(input_dim,)))

    # Batch normalise the input
    model.add(layers.BatchNormalization())

    for i in range(hidden_layers):
        n = neurons[i] if i < len(neurons) else neurons[-1]
        model.add(layers.Dense(n, activation=activation))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))

    # sigmoid for binary classification
    model.add(layers.Dense(1, activation='sigmoid'))

    return model


def get_optimizer(name, lr):
    if name == 'sgd':
        return keras.optimizers.SGD(learning_rate=lr)
    elif name == 'adam':
        return keras.optimizers.Adam(learning_rate=lr)
    elif name == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=lr)
    elif name == 'adamax':
        return keras.optimizers.Adamax(learning_rate=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {name}")


def train_experiment(exp, X_train, y_train, X_test, y_test, class_weight_dict):
    set_seed(42)
    keras.backend.clear_session()
    header_format(f"Running {exp['name']}...")
    input_dim = X_train.shape[1]
    model = build_model(
        input_dim=input_dim,
        hidden_layers=exp['hidden_layers'],
        neurons=exp['neurons'],
        activation=exp['activation'],
        dropout_rate=exp['dropout_rate']
    )

    model.compile(
        optimizer=get_optimizer(exp['optimizer'], exp['lr']),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            'accuracy'
        ]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            patience=10,
            restore_best_weights=True,
            mode='max'
        )
    ]

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=exp['epochs'],
        batch_size=exp['batch_size'],
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=0
    )
    end_time = time.time()
    elapsed  = end_time - start_time


    best_ep = int(np.argmax(history.history['val_auc']))
    best_train_acc  = history.history['accuracy'][best_ep]
    best_train_auc  = history.history['auc'][best_ep]
    best_train_prec = history.history['precision'][best_ep]
    best_train_rec  = history.history['recall'][best_ep]
    test_loss, test_auc, test_prec, test_rec, test_acc = model.evaluate(
        X_test, y_test, verbose=0)

    epochs_run = len(history.history['loss'])
    gap        = abs(best_train_acc - test_acc)

    return (model,
            best_train_acc, best_train_auc, best_train_prec, best_train_rec,
            test_acc, test_auc, test_prec, test_rec,
            epochs_run, elapsed, history, gap)


def run_experiments(experiments, X_train, y_train, X_test, y_test, class_weight_dict, plot_history=True):
    results         = []
    best_experiment = None

    for idx, exp in enumerate(experiments, 1):
        (model,
         train_acc, train_auc, train_prec, train_rec,
         test_acc,  test_auc,  test_prec,  test_rec,
         epochs_run, elapsed, history, gap) = train_experiment(
            exp, X_train, y_train, X_test, y_test, class_weight_dict)

        results.append({
            'experiment': exp['name'],
            'train_acc':  train_acc,
            'train_auc':  train_auc,
            'train_prec': train_prec,
            'train_rec':  train_rec,
            'test_acc':   test_acc,
            'test_auc':   test_auc,
            'test_prec':  test_prec,
            'test_rec':   test_rec,
            'epochs_run': epochs_run,
            'elapsed':    elapsed,
            'gap':        gap
        })

        if plot_history:
            plot_training_history(history, title_suffix=f"({exp['name']})")

    # Best experiment by test_auc — most reliable metric
    best_result = get_best_result(results) #max(results, key=lambda x: x['test_auc'])
    best_experiment = next(
        exp for exp in experiments if exp['name'] == best_result['experiment'])

    header_format("Experiments Results")
    print(f"{'#':<3} {'Experiment':<55} "
          f"{'Train Acc':>10} "
          f"{'Test Acc':>10} "
          f"{'Test AUC':>10} "
          f"{'Precision':>10} "
          f"{'Recall':>10} "
          f"{'Gap':>8} "
          f"{'Epochs':>7} "
          f"{'Time':>8}")
    print(f"{'-'*145}")

    for idx, result in enumerate(results, 1):
        print_row(
            idx,
            result['experiment'],
            result['train_acc'],
            result['test_acc'],
            result['test_auc'],
            result['test_prec'],
            result['test_rec'],
            result['epochs_run'],
            result['elapsed'],
            result['gap'],
            best=result['experiment'] == best_result['experiment']
        )

    return model, best_result, best_experiment

In [ ]:
target_column = 'FirstYearPersistence'
print(f"{target_column} Target Distribution:")
print(df[target_column].value_counts())
print(df[target_column].value_counts(normalize=True))


In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

# Attempt to handle class imbalance with class weights
classes = np.unique(y)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weight_dict = dict(zip(classes, class_weights))

print(f"\nClasses: {classes}")
print(f"\nClass Weights: {class_weight_dict}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain persistence rate: {y_train.mean():.2%}")
print(f"Test persistence rate:  {y_test.mean():.2%}")

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
a1_model, a1_best_architecture_results, a1_best_architecture_exp_parameters = run_experiments(experiments, X_train_scaled, y_train, X_test_scaled, y_test, class_weight_dict, plot_history=True)

## Approach #2: Drop All Rows with Missing Values

In [ ]:
print(df_original.isnull().sum())

In [ ]:
print(f"Total rows:           {len(df_original)}")
print(f"Rows with no missing: {df_original.dropna().shape[0]}")
print(f"Rows dropped:         {len(df_original) - df_original.dropna().shape[0]}")
print(f"Percentage lost:      {(len(df_original) - df_original.dropna().shape[0]) / len(df_original) * 100:.2f}%")

df_dropped = df_original.dropna().copy()
print(f"\nTarget distribution after dropping:")
print(df_dropped['FirstYearPersistence'].value_counts())
print(df_dropped['FirstYearPersistence'].value_counts(normalize=True))
print(f"Shape after dropping: {df_dropped.shape}")

In [ ]:
X_drop = df_dropped.drop(columns=['FirstYearPersistence', 'hs_avg_missing', 'math_score_missing'])
y_drop = df_dropped['FirstYearPersistence']

classes_drop    = np.unique(y_drop)
weights_drop    = compute_class_weight(class_weight='balanced', classes=classes_drop, y=y_drop)
cw_drop         = dict(zip(classes_drop, weights_drop))

print(f"\nClass weights: {cw_drop}")
print(f"Dropout weight: {cw_drop[0]:.4f}")
print(f"Persist weight: {cw_drop[1]:.4f}")

In [ ]:
X_train_drop, X_test_drop, y_train_drop, y_test_drop = train_test_split(X_drop, y_drop, test_size=0.2, random_state=42, stratify=y_drop)
print(f"Train persistence rate: {y_train_drop.mean():.2%}")
print(f"Test persistence rate:  {y_test_drop.mean():.2%}")

In [ ]:
scaler_drop      = MinMaxScaler()
X_train_drop_sc  = scaler_drop.fit_transform(X_train_drop)
X_test_drop_sc   = scaler_drop.transform(X_test_drop)
print(f"Input features: {X_train_drop_sc.shape[1]}")

In [ ]:
a2_model, a2_best_architecture_results, a2_best_architecture_exp_parameters = run_experiments(experiments, X_train_drop_sc, y_train_drop, X_test_drop_sc, y_test_drop, cw_drop, plot_history=True)

## Approach #3: Drop high missing columns + simple median/mode for remaining columns

In [ ]:
cols_to_drop = ['High School Average Mark', 'Math Score', 'hs_avg_missing', 'math_score_missing']
continuous_cols = ['First Term Gpa', 'Second Term Gpa']
categorical_cols = ['First Language', 'Previous Education', 
                    'Age Group', 'English Grade']

df_col_dropped = df_original.drop(columns=cols_to_drop).copy()

print(f"Original shape:      {df_original.shape}")
print(f"After drop shape:    {df_col_dropped.shape}")
print(f"Columns removed:     {cols_to_drop}")

print(f"\nRemaining missing values:")
print(df_col_dropped.isnull().sum())
print(f"\nTotal missing: {df_col_dropped.isnull().sum().sum()}")

print(f"\nTarget distribution:")
print(df_col_dropped['FirstYearPersistence'].value_counts())
print(df_col_dropped['FirstYearPersistence'].value_counts(normalize=True))

In [ ]:
median_imp = SimpleImputer(strategy='median')
df_col_dropped[continuous_cols] = median_imp.fit_transform(df_col_dropped[continuous_cols])

mode_imp = SimpleImputer(strategy='most_frequent')
df_col_dropped[categorical_cols] = mode_imp.fit_transform(df_col_dropped[categorical_cols])

print(f"Missing after imputation: {df_col_dropped.isnull().sum().sum()}")
print(f"\nFinal shape: {df_col_dropped.shape}")
print(f"\nColumns used:\n{df_col_dropped.columns.tolist()}")

In [ ]:
X_drop_high_mis_col = df_col_dropped.drop(columns=['FirstYearPersistence'])
y_drop_high_mis_col = df_col_dropped['FirstYearPersistence']

classes_drop_high_mis_col  = np.unique(y_drop_high_mis_col)
weights_drop_high_mis_col  = compute_class_weight(class_weight='balanced', 
                                    classes=classes_drop_high_mis_col, 
                                    y=y_drop_high_mis_col)
cw_drop_high_mis_col = dict(zip(classes_drop_high_mis_col, weights_drop_high_mis_col))
print(f"\nClass weights: {cw_drop_high_mis_col}")

X_train_drop_high_mis_col, X_test_drop_high_mis_col, y_train_drop_high_mis_col, y_test_drop_high_mis_col = train_test_split(
    X_drop_high_mis_col, y_drop_high_mis_col,
    test_size=0.2,
    random_state=42,
    stratify=y_drop_high_mis_col
)

scaler_drop_high_mis_col = MinMaxScaler()
X_train_drop_high_mis_col_sc = scaler_drop_high_mis_col.fit_transform(X_train_drop_high_mis_col)
X_test_drop_high_mis_col_sc = scaler_drop_high_mis_col.transform(X_test_drop_high_mis_col)

a3_model, a3_best_architecture_results, a3_best_architecture_exp_parameters = run_experiments(experiments, X_train_drop_high_mis_col_sc, y_train_drop_high_mis_col, X_test_drop_high_mis_col_sc, y_test_drop_high_mis_col, cw_drop_high_mis_col, plot_history=True)

## Approach #4: Simple Imputation for ALL columns

In [ ]:
df_simple_imputation = df_original.copy()
df_simple_imputation = df_simple_imputation.drop(
    columns=['hs_avg_missing', 'math_score_missing'], 
    errors='ignore'
)

continuous_cols  = ['First Term Gpa', 'Second Term Gpa', 'High School Average Mark', 'Math Score']
categorical_cols = ['First Language', 'Previous Education', 'Age Group', 'English Grade']

median_imp = SimpleImputer(strategy='median')
df_simple_imputation[continuous_cols] = median_imp.fit_transform(
    df_simple_imputation[continuous_cols])

mode_imp = SimpleImputer(strategy='most_frequent')
df_simple_imputation[categorical_cols] = mode_imp.fit_transform(
    df_simple_imputation[categorical_cols])

print(f"Shape: {df_simple_imputation.shape}")
print(f"Total missing: {df_simple_imputation.isnull().sum().sum()}")
print(f"\nMissing per column:")
print(df_simple_imputation.isnull().sum())

In [ ]:
X_simple_imputation = df_simple_imputation.drop(columns=['FirstYearPersistence'])
y_simple_imputation = df_simple_imputation['FirstYearPersistence']

classes_simple_imputation = np.unique(y_simple_imputation)
weights_simple_imputation = compute_class_weight(class_weight='balanced',
                                     classes=classes_simple_imputation,
                                     y=y_simple_imputation)
cw_simple_imputation = dict(zip(classes_simple_imputation, weights_simple_imputation))
print(f"\nClass weights: {cw_simple_imputation}")

X_train_simple_imputation, X_test_simple_imputation, y_train_simple_imputation, y_test_simple_imputation = train_test_split(
    X_simple_imputation, y_simple_imputation,
    test_size=0.2,
    random_state=42,
    stratify=y_simple_imputation
)

scaler_simple_imputation = MinMaxScaler()
X_train_simple_imputation_sc = scaler_simple_imputation.fit_transform(X_train_simple_imputation)
X_test_simple_imputation_sc = scaler_simple_imputation.transform(X_test_simple_imputation)

a4_model, a4_best_architecture_results, a4_best_architecture_exp_parameters = run_experiments(experiments, X_train_simple_imputation_sc, y_train_simple_imputation, X_test_simple_imputation_sc, y_test_simple_imputation, cw_simple_imputation)

## Approach #5: Single Iterative Imputer

In [ ]:
df_single_iterative_imputer = df_original.copy()

# Drop missingness flags if they exist
df_single_iterative_imputer = df_single_iterative_imputer.drop(
    columns=['hs_avg_missing', 'math_score_missing'],
    errors='ignore'
)

# Define all columns to impute
all_cols = df_single_iterative_imputer.drop(columns=['FirstYearPersistence']).columns.tolist()

print(f"Columns to impute: {all_cols}")
print(f"\nMissing before imputation:")
print(df_single_iterative_imputer.isnull().sum())

# Apply single IterativeImputer to all columns at once  
iter_imp = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    tol=1e-2,
    random_state=42
)

# Fit and transform all feature columns
df_iter_cols = df_single_iterative_imputer[all_cols].copy()
df_iter_imputed = pd.DataFrame(
    iter_imp.fit_transform(df_iter_cols),
    columns=all_cols,
    index=df_iter_cols.index
)

# Fill imputed values back
df_single_iterative_imputer[all_cols] = df_iter_imputed

# Round categorical columns back to valid integers
categorical_cols = ['First Language', 'Previous Education',
                    'Age Group', 'English Grade']

for col in categorical_cols:
    df_single_iterative_imputer[col] = df_single_iterative_imputer[col].round().clip(
        df_original[col].min(),
        df_original[col].max()
    ).astype(int)

print(f"\nShape: {df_single_iterative_imputer.shape}")
print(f"Total missing: {df_single_iterative_imputer.isnull().sum().sum()}")
print(f"\nMissing per column:")
print(df_single_iterative_imputer.isnull().sum())

In [ ]:
X_single_iterative_imputer = df_single_iterative_imputer.drop(columns=['FirstYearPersistence'])
y_single_iterative_imputer = df_single_iterative_imputer['FirstYearPersistence']

classes_single_iterative_imputer = np.unique(y_single_iterative_imputer)
weights_single_iterative_imputer = compute_class_weight(class_weight='balanced', classes=classes_single_iterative_imputer, y=y_single_iterative_imputer)
cw_single_iterative_imputer = dict(zip(classes_single_iterative_imputer, weights_single_iterative_imputer))
print(f"\nClass weights: {cw_single_iterative_imputer}")

X_train_single_iterative_imputer, X_test_single_iterative_imputer, y_train_single_iterative_imputer, y_test_single_iterative_imputer = train_test_split(
    X_single_iterative_imputer, y_single_iterative_imputer,
    test_size=0.2,
    random_state=42,
    stratify=y_single_iterative_imputer
)

scaler_single_iterative_imputer = MinMaxScaler()
X_train_single_iterative_imputer_sc = scaler_single_iterative_imputer.fit_transform(X_train_single_iterative_imputer)
X_test_single_iterative_imputer_sc = scaler_single_iterative_imputer.transform(X_test_single_iterative_imputer)
a5_model, a5_best_architecture_results, a5_best_architecture_exp_parameters = run_experiments(experiments, X_train_single_iterative_imputer_sc, y_train_single_iterative_imputer, X_test_single_iterative_imputer_sc, y_test_single_iterative_imputer, cw_single_iterative_imputer, plot_history=True)

## Approach #6: Adding Missingness Flags

In [ ]:
df_missing_flags = df_original.copy()
df_missing_flags = df_missing_flags.drop(
    columns=['hs_avg_missing', 'math_score_missing'],
    errors='ignore'
)

missing_cols = df_missing_flags.columns[df_missing_flags.isnull().any()].tolist()
missing_cols = [c for c in missing_cols if c != 'FirstYearPersistence']

print(f"Columns with missing values:")
for col in missing_cols:
    count = df_missing_flags[col].isnull().sum()
    pct   = count / len(df_missing_flags) * 100
    print(f"  {col:<30} {count:>4} rows ({pct:.2f}%)")

In [ ]:
for col in missing_cols:
    flag_col = col.lower().replace(' ', '_') + '_missing'
    df_missing_flags[flag_col] = df_missing_flags[col].isnull().astype(int)
    print(f"Added flag: {flag_col} — {df_missing_flags[flag_col].sum()} flagged rows")

df_missing_flags[missing_cols] = df_missing_flags[missing_cols].fillna(0)

In [ ]:
X_missing_flags = df_missing_flags.drop(columns=['FirstYearPersistence'])
y_missing_flags = df_missing_flags['FirstYearPersistence']
classes_missing_flags   = np.unique(y_missing_flags)
weights_missing_flags   = compute_class_weight(class_weight='balanced',
                                               classes=classes_missing_flags,
                                               y=y_missing_flags)
cw_missing_flags = dict(zip(classes_missing_flags, weights_missing_flags))
print(f"\nClass weights: {cw_missing_flags}")

X_train_missing_flags, X_test_missing_flags, y_train_missing_flags, y_test_missing_flags = train_test_split(
    X_missing_flags, y_missing_flags,
    test_size=0.2,
    random_state=42,
    stratify=y_missing_flags
)

scaler_missing_flags     = MinMaxScaler()
X_train_missing_flags_sc = scaler_missing_flags.fit_transform(X_train_missing_flags)
X_test_missing_flags_sc  = scaler_missing_flags.transform(X_test_missing_flags)

a6_model, a6_best_architecture_results, a6_best_architecture_exp_parameters = run_experiments(experiments, X_train_missing_flags_sc, y_train_missing_flags, X_test_missing_flags_sc, y_test_missing_flags, cw_missing_flags, plot_history=True)